# CLIP Prompt and Candidate Sets

Change the recorded prompt set. Does a high score remain high after adding a candidate?

Run the setup cells, then change the input in **Demo code**. All packages, models and data stay in the Colab runtime. For large models, select a GPU runtime. Free GPU availability varies.

Source: https://huggingface.co/openai/clip-vit-base-patch32

## Runtime setup

In [ ]:
# Packages are installed in the Colab runtime, not on your computer.
%pip -q install "matplotlib>=3.9,<3.12" "scikit-learn==1.6.1" "Pillow" "requests" "tqdm" "transformers==4.57.1" "huggingface-hub==0.36.2" "accelerate" "safetensors"
import os
os.environ["TABPFN_DISABLE_TELEMETRY"] = "1"
os.environ["WANDB_MODE"] = "disabled"
import torch
print("Runtime:", "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import os
import random
import tarfile
import requests
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from transformers import (
    BertTokenizer, BertModel, BertForMaskedLM,
    CLIPProcessor, CLIPModel, CLIPTokenizerFast,
)
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

In [ ]:
CLIP_ID = "openai/clip-vit-base-patch32"

print(f"📦 Loading CLIP: {CLIP_ID}")
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
clip_model = CLIPModel.from_pretrained(CLIP_ID, revision="refs/pr/15", use_safetensors=True).to(device)
clip_model.eval()

clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"   ✅ Loaded!")
print(f"   - Total parameters: {clip_params / 1e6:.1f}M")
print(f"   - Vision (ViT): {sum(p.numel() for p in clip_model.vision_model.parameters())/1e6:.1f}M")
print(f"   - Text:          {sum(p.numel() for p in clip_model.text_model.parameters())/1e6:.1f}M")
print(f"   - Image size:    {clip_model.config.vision_config.image_size}")
print(f"   - Patch size:    {clip_model.config.vision_config.patch_size}")
print(f"   - Embed dim:     {clip_model.config.projection_dim}")

## Input images

Fixed Oxford-IIIT Pet classroom subset. Source: https://www.robots.ox.ac.uk/~vgg/data/pets/

In [ ]:
from pathlib import Path
import urllib.request, zipfile

IMG_DIR = "./data"
NUM_IMAGES_PER_CLASS = 50
batch_size = 16
if not all(len(list((Path(IMG_DIR) / label).glob("*.jpg"))) == 50 for label in ["dog", "cat"]):
    url = "https://moai.postech.ac.kr/teaching/27s-programming-for-ax/labs/colab/oxford-pets-classroom.zip"
    archive, _ = urllib.request.urlretrieve(url)
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(".")
image_paths, image_labels = [], []
for label in ["dog", "cat"]:
    paths = sorted((Path(IMG_DIR) / label).glob("*.jpg"))
    image_paths.extend(str(path) for path in paths)
    image_labels.extend([label] * len(paths))
print(f"Classroom subset: {len(image_paths)} images")


In [ ]:
@torch.no_grad()
def clip_zero_shot_classify(images, class_texts):
    """
    Perform zero-shot classification with CLIP.

    Args:
        images: list of PIL Image
        class_texts: list of str (e.g., ["a photo of a dog", "a photo of a cat"])

    Returns:
        probs: (N, C) probability matrix
        pred_indices: (N,) predicted class indices
    """
    inputs = clip_processor(
        text=class_texts, images=images,
        return_tensors="pt", padding=True
    ).to(device)

    outputs = clip_model(**inputs)
    logits = outputs.logits_per_image  # (N, C)
    probs = logits.softmax(dim=1).cpu().numpy()
    pred_indices = probs.argmax(axis=1)

    return probs, pred_indices

print("✅ clip_zero_shot_classify function defined")

In [ ]:
@torch.no_grad()
def get_clip_image_embeddings(images):
    """Extract image embeddings with the CLIP image encoder."""
    inputs = clip_processor(images=images, return_tensors="pt").to(device)
    embeds = clip_model.get_image_features(**inputs)
    return F.normalize(embeds, p=2, dim=1).cpu().numpy()

@torch.no_grad()
def get_clip_text_embeddings(texts):
    """Extract text embeddings with the CLIP text encoder."""
    inputs = clip_processor(text=texts, return_tensors="pt", padding=True).to(device)
    embeds = clip_model.get_text_features(**inputs)
    return F.normalize(embeds, p=2, dim=1).cpu().numpy()

# Select sample images (4 dogs, 4 cats)
sample_dog_paths = image_paths[:4]
sample_cat_paths = image_paths[NUM_IMAGES_PER_CLASS:NUM_IMAGES_PER_CLASS+4]
sample_paths = sample_dog_paths + sample_cat_paths
sample_images = [Image.open(p).convert('RGB') for p in sample_paths]

# Text prompts
text_prompts = [
    "a photo of a dog",
    "a puppy playing",
    "a photo of a cat",
    "a kitten sleeping",
    "a bird flying",
    "a photo of food",
]

# Extract embeddings
img_embeds = get_clip_image_embeddings(sample_images)
txt_embeds = get_clip_text_embeddings(text_prompts)

# Image-Text Similarity Matrix
similarity = img_embeds @ txt_embeds.T

fig = plt.figure(figsize=(18, 8))
gs = fig.add_gridspec(2, len(sample_images) + 1, width_ratios=[1]*len(sample_images) + [6])

# Image thumbnails (left)
for i, img in enumerate(sample_images):
    ax = fig.add_subplot(gs[i // 4, i % 4])
    ax.imshow(img)
    label = 'Dog' if i < 4 else 'Cat'
    ax.set_title(f'{label} #{i % 4}', fontsize=8)
    ax.axis('off')

# Similarity heatmap (right)
ax_heatmap = fig.add_subplot(gs[:, -1])
im = ax_heatmap.imshow(similarity, cmap='RdYlBu_r', aspect='auto')

img_labels_show = [f"Dog #{i}" for i in range(4)] + [f"Cat #{i}" for i in range(4)]
ax_heatmap.set_yticks(range(len(img_labels_show)))
ax_heatmap.set_yticklabels(img_labels_show, fontsize=10)
ax_heatmap.set_xticks(range(len(text_prompts)))
ax_heatmap.set_xticklabels(text_prompts, rotation=30, ha='right', fontsize=9)
ax_heatmap.set_title('CLIP Image-Text Similarity', fontsize=14, fontweight='bold')

for i in range(len(sample_images)):
    for j in range(len(text_prompts)):
        ax_heatmap.text(j, i, f'{similarity[i,j]:.2f}', ha='center', va='center', fontsize=9,
                       color='white' if similarity[i,j] > 0.25 else 'black')

ax_heatmap.axhline(y=3.5, color='black', linewidth=2)
plt.colorbar(im, ax=ax_heatmap, shrink=0.6)

plt.tight_layout()
plt.show()

print("💡 Observation: dog images have high similarity to 'dog' text; cat images have high similarity to 'cat' text.")
print("   Similarity to unrelated text ('bird', 'food') is low.")

## Demo code

Change the recorded prompt set. Does a high score remain high after adding a candidate?

In [ ]:
images = [Image.open(image_paths[0]).convert("RGB")]
candidates = ["a photo of a dog", "a photo of a cat", "a photo of a bird"]
probabilities, predictions = clip_zero_shot_classify(images, candidates)
print(dict(zip(candidates, probabilities[0].round(3))))
print("Top candidate:", candidates[predictions[0]])

In [ ]:
# Demo zero-shot classification on sample images
class_texts = ["a photo of a dog", "a photo of a cat"]
n_show = 10
show_indices = random.sample(range(len(image_paths)), n_show)

fig, axes = plt.subplots(2, n_show // 2, figsize=(4 * (n_show // 2), 8))
fig.suptitle('🎯 CLIP Zero-Shot Classification (no training!)', fontsize=16, fontweight='bold')

for i, idx in enumerate(show_indices):
    ax = axes[i // (n_show // 2), i % (n_show // 2)]
    img = Image.open(image_paths[idx]).convert('RGB')
    gt = image_labels[idx]

    probs, pred_idx = clip_zero_shot_classify([img], class_texts)
    pred = 'dog' if pred_idx[0] == 0 else 'cat'
    correct = pred == gt

    ax.imshow(img)
    symbol = '✓' if correct else '✗'
    color = 'green' if correct else 'red'
    ax.set_title(f'{symbol} GT: {gt.upper()} | Pred: {pred.upper()}\n'
                f'Dog: {probs[0][0]:.1%}  Cat: {probs[0][1]:.1%}',
                fontsize=10, fontweight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Measure zero-shot classification accuracy on the entire dataset
print("📊 Running CLIP zero-shot classification on the full dataset...\n")

class_texts = ["a photo of a dog", "a photo of a cat"]
all_preds = []
all_probs = []

batch_size = 16
for i in tqdm(range(0, len(image_paths), batch_size)):
    batch_paths = image_paths[i:i+batch_size]
    batch_images = [Image.open(p).convert('RGB') for p in batch_paths]

    probs, pred_idx = clip_zero_shot_classify(batch_images, class_texts)
    all_probs.extend(probs.tolist())
    all_preds.extend(['dog' if p == 0 else 'cat' for p in pred_idx])

acc = accuracy_score(image_labels, all_preds)
cm = confusion_matrix(image_labels, all_preds, labels=['cat', 'dog'])

print(f"\n{'='*50}")
print(f"  🎯 CLIP Zero-Shot Accuracy: {acc:.1%}")
print(f"  (achieved with 0 training examples!)")
print(f"{'='*50}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0, 1]); axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(['Cat', 'Dog']); axes[0].set_yticklabels(['Cat', 'Dog'])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title(f'Zero-Shot Confusion Matrix\nAccuracy: {acc:.1%}', fontsize=13, fontweight='bold')
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=20, fontweight='bold',
                    color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0], shrink=0.8)

# Confidence distribution
correct_confs = [max(p) for p, gt, pred in zip(all_probs, image_labels, all_preds) if gt == pred]
wrong_confs = [max(p) for p, gt, pred in zip(all_probs, image_labels, all_preds) if gt != pred]
axes[1].hist(correct_confs, bins=20, alpha=0.7, color='green', label=f'Correct ({len(correct_confs)})', density=True)
if wrong_confs:
    axes[1].hist(wrong_confs, bins=20, alpha=0.7, color='red', label=f'Wrong ({len(wrong_confs)})', density=True)
axes[1].set_xlabel('Max Probability'); axes[1].set_ylabel('Density')
axes[1].set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Compare various prompt templates
prompt_templates = {
    'Simple':       ['dog', 'cat'],
    'A photo of':   ['a photo of a dog', 'a photo of a cat'],
    'Descriptive':  ['a cute dog, a pet animal', 'a cute cat, a pet animal'],
    'This is':      ['this is a dog', 'this is a cat'],
    'Animal type':  ['a dog, which is a canine', 'a cat, which is a feline'],
    'Negative':     ['not a cat, this is a dog', 'not a dog, this is a cat'],
}

print("🧪 Zero-shot accuracy comparison across prompt templates\n")

results_by_template = {}

# Process all images at once
all_images = [Image.open(p).convert('RGB') for p in image_paths]

for name, prompts in prompt_templates.items():
    preds = []
    for i in range(0, len(all_images), batch_size):
        batch = all_images[i:i+batch_size]
        probs, pred_idx = clip_zero_shot_classify(batch, prompts)
        preds.extend(['dog' if p == 0 else 'cat' for p in pred_idx])

    acc = accuracy_score(image_labels, preds)
    results_by_template[name] = acc
    print(f"  {name:15s}  {prompts[0]:40s}  →  {acc:.1%}")

# Visualization
fig, ax = plt.subplots(figsize=(10, 5))
names = list(results_by_template.keys())
accs = list(results_by_template.values())
colors = ['#4ECDC4' if a == max(accs) else '#95A5A6' for a in accs]

bars = ax.bar(names, accs, color=colors, edgecolor='black', alpha=0.8)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('📝 Zero-Shot Accuracy by Prompt Template', fontsize=14, fontweight='bold')
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
ax.legend()

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
           f'{acc:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

best = max(results_by_template, key=results_by_template.get)
print(f"\n💡 Best prompt: '{best}' ({results_by_template[best]:.1%})")
print("   The CLIP paper also reports that the form 'a photo of a {class}' is most effective.")

## Interpretation

Softmax scores depend on the candidate set.

Compare the output with the evidence in the input. A generated answer can be wrong.